This notebook has been used to generate some first tests using GX. It is based on the GX getting started docs. It is not intended to be run externally but only for generating the json configuration which can then be executed from CRON or Airflow.

In [2]:
import great_expectations
from great_expectations.core import ExpectationConfiguration
context = great_expectations.get_context()
import logging

In [3]:
import os
gx_context_root_dir=os.environ['GX_CONTEXT_ROOT_DIR']

In [4]:
import yaml

In [5]:
from datetime import date,datetime

In [6]:
logging.basicConfig(level=logging.WARN, force = True)

In [7]:
with open(f"{gx_context_root_dir}/datasources/datasources.yml", "r") as ymlfile:
    datasource_config = yaml.full_load(ymlfile)

In [8]:
datasource_config.get("project")

'gfw-google-827'

In [9]:
gx_project = datasource_config.get("project")
gx_datasource = context.get_datasource(gx_project)

In [10]:
gx_datasource.get_asset_names()

{'encounters-2.5',
 'encounters-3.0.0',
 'features_-2.5',
 'features_-3.0.0',
 'fishing_score_-2.5',
 'fishing_score_-3.0.0',
 'fragments-3.0.0',
 'messages-2.5',
 'messages-3.0.0',
 'messages_positions-2.5',
 'messages_positions-3.0.0',
 'messages_scored_-2.5',
 'messages_scored_-3.0.0',
 'messages_segmented_-2.5',
 'messages_segmented_-3.0.0',
 'satellite_timing_offsets-2.5',
 'satellite_timing_offsets-3.0.0',
 'segment_identity_daily_-2.5',
 'segment_identity_daily_-3.0.0',
 'segment_info-2.5',
 'segment_info-3.0.0',
 'segment_vessel-2.5',
 'segment_vessel-3.0.0',
 'segment_vessel_daily_-2.5',
 'segment_vessel_daily_-3.0.0',
 'segments-2.5',
 'segments-3.0.0',
 'segs_activity-2.5',
 'segs_activity-3.0.0',
 'segs_activity_daily-2.5',
 'segs_activity_daily-3.0.0',
 'ssvids_identities-2.5',
 'ssvids_identities-3.0.0',
 'ssvids_identities_daily-2.5',
 'ssvids_identities_daily-3.0.0',
 'stats_daily-2.5',
 'stats_daily-3.0.0',
 'vessel_info-2.5',
 'vessel_info-3.0.0'}

In [11]:
context.list_expectation_suite_names()

['gfw-google-827.alerts.encounters.2-5',
 'gfw-google-827.alerts.encounters.3-0-0',
 'gfw-google-827.alerts.features_.2-5',
 'gfw-google-827.alerts.features_.3-0-0',
 'gfw-google-827.alerts.fishing_score_.2-5',
 'gfw-google-827.alerts.fishing_score_.3-0-0',
 'gfw-google-827.alerts.fragments.3-0-0',
 'gfw-google-827.alerts.messages.2-5',
 'gfw-google-827.alerts.messages.3-0-0',
 'gfw-google-827.alerts.messages_positions.2-5',
 'gfw-google-827.alerts.messages_positions.3-0-0',
 'gfw-google-827.alerts.messages_scored_.2-5',
 'gfw-google-827.alerts.messages_scored_.3-0-0',
 'gfw-google-827.alerts.messages_segmented_.2-5',
 'gfw-google-827.alerts.messages_segmented_.3-0-0',
 'gfw-google-827.alerts.satellite_timing_offsets.2-5',
 'gfw-google-827.alerts.satellite_timing_offsets.3-0-0',
 'gfw-google-827.alerts.segment_identity_daily_.2-5',
 'gfw-google-827.alerts.segment_identity_daily_.3-0-0',
 'gfw-google-827.alerts.segment_info.2-5',
 'gfw-google-827.alerts.segment_info.3-0-0',
 'gfw-google

In [13]:
current_asset_name = 'segments.2-5'
current_asset_constraints=[es for es in context.list_expectation_suite_names() if current_asset_name in es and 'constraints' in es]
current_asset_constraints

['gfw-google-827.constraints.segments.2-5']

In [16]:
for current_expectation_suite_name in current_asset_constraints:
    print(current_expectation_suite_name)
    current_expectation_suite=context.get_expectation_suite(current_expectation_suite_name)    
    current_expectation_suite_asset_name=current_expectation_suite.meta.get('asset_name')
    current_expectation_suite_datasource_name=current_expectation_suite.meta.get('datasource_name')
    current_expectation_suite_version_number=current_expectation_suite.meta.get('version_number')

    gx_asset=gx_datasource.get_asset(current_expectation_suite_asset_name)
    gx_splitter=gx_asset.splitter
    if gx_splitter is not None:
        DATE_PARTITION_COLUMN=gx_splitter.column_name
        br_options={DATE_PARTITION_COLUMN: '2023-04-01'}
    else:
        br_options={}
    gx_br = gx_asset.build_batch_request(br_options)

    gx_batches = gx_datasource.get_batch_list_from_batch_request(gx_br)

    gx_validator = context.get_validator_using_batch_list(current_expectation_suite, gx_batches)

    gx_validator.expect_column_values_to_be_unique('seg_id')
    gx_validator.expect_column_values_to_not_be_null('seg_id')

    gx_validator.expect_column_min_to_be_between('message_count', min_value=0, strict_min=True)

    gx_validator.expect_queried_custom_query_to_return_num_rows(template_dict={"user_query": f"""
        SELECT *
        FROM {{active_batch}}
        WHERE LEFT(seg_id, STRPOS(seg_id, "-")-1) != ssvid
    """}, value=0, meta={
                "notes": {
                    "format": "markdown",
                    "content": "The `seg_id` should start with the `ssvid`.",
            }
    })

    gx_validator.save_expectation_suite(discard_failed_expectations=False)

gfw-google-827.constraints.segments.2-5


Calculating Metrics:   0%|          | 0/10 [00:00<?, ?it/s]

Calculating Metrics:   0%|          | 0/8 [00:00<?, ?it/s]

Calculating Metrics:   0%|          | 0/6 [00:00<?, ?it/s]

  warnings.warn(str(e), UserWarning)



Calculating Metrics:   0%|          | 0/1 [00:00<?, ?it/s]